In [1]:
# ===============================
# 0. Dependencies & Setup
# ===============================
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install -q torch_geometric matplotlib tqdm scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import matplotlib.pyplot as plt
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH     = '/content/drive/MyDrive/Honors/'   # UPDATE THIS IF NEEDED
AUDIO_FEAT    = 'audio_features_imp.npy'
VIDEO_FEAT    = 'video_features_imp.npy'
CONTEXT_FEAT  = 'context_features_imp.npy'
LABELS_FEAT   = 'labels_imp.npy'

BATCH_SIZE    = 16
EPOCHS        = 100
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 1e-4
PATIENCE      = 15

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.7 MB/s eta 0:00:00
Using device: cuda


In [2]:
# ===============================
# 1. Load Features
# ===============================
from google.colab import drive
drive.mount('/content/drive')

print("Loading Audio & Video Features...")

a_features = np.load(os.path.join(BASE_PATH, AUDIO_FEAT))
v_features = np.load(os.path.join(BASE_PATH, VIDEO_FEAT))
c_features = np.load(os.path.join(BASE_PATH, CONTEXT_FEAT))
labels     = np.load(os.path.join(BASE_PATH, LABELS_FEAT))

print(f"Loaded {len(labels)} samples.")
print(f"Dims: A={a_features.shape[1]}, V={v_features.shape[1]}, C={c_features.shape[1]}")

Mounted at /content/drive
Loading Audio & Video Features...
Loaded 690 samples.
Dims: A=768, V=2048, C=2816


In [3]:
# ===============================
# 2. Build Graph Dataset
# ===============================
data_list = []
for i in range(len(labels)):
    a = torch.tensor(a_features[i], dtype=torch.float).unsqueeze(0)
    v = torch.tensor(v_features[i], dtype=torch.float).unsqueeze(0)
    c = torch.tensor(c_features[i], dtype=torch.float).unsqueeze(0)
    y = torch.tensor([labels[i]], dtype=torch.float)

    # Nodes: 0=A, 1=V, 2=C
    edge_index = torch.tensor([
        [0,1],[1,0],   # Audio <-> Video
        [2,0],[0,2],   # Context <-> Audio
        [2,1],[1,2],   # Context <-> Video
        [0,0],[1,1],[2,2] # Self loops
    ], dtype=torch.long).t().contiguous()

    data = Data(edge_index=edge_index, y=y)
    data.audio   = a
    data.video   = v
    data.context = c
    data.num_nodes = 3

    data_list.append(data)

print(f"Built {len(data_list)} graph samples.")

Built 690 graph samples.


In [4]:
# ===============================
# 3. Train/Val/Test Split
# ===============================
train_val, test = train_test_split(
    data_list, test_size=0.15, random_state=42, stratify=labels
)

labels_train_val = np.array([d.y.item() for d in train_val])
train, val = train_test_split(
    train_val, test_size=0.1765, random_state=42, stratify=labels_train_val
)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test,  batch_size=BATCH_SIZE, shuffle=False)

Train: 482, Val: 104, Test: 104


In [5]:
# ===============================
# 4. Audio-Video SarcasmGATv2 Model
# ===============================
class AudioVideoSarcasmGATv2(nn.Module):
    def __init__(self, a_dim, v_dim, c_dim, hidden_dim=256, gat_dropout=0.5, fc_dropout=0.5):
        super().__init__()

        # Projections
        self.p_a = nn.Sequential(nn.Linear(a_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_v = nn.Sequential(nn.Linear(v_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_c = nn.Sequential(nn.Linear(c_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        # GATv2
        self.gat1 = GATv2Conv(hidden_dim, hidden_dim, heads=8, concat=True, dropout=gat_dropout)
        self.gat2 = GATv2Conv(hidden_dim * 8, hidden_dim, heads=1, concat=False, dropout=gat_dropout)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(fc_dropout),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        x_a = self.p_a(data.audio)
        x_v = self.p_v(data.video)
        x_c = self.p_c(data.context)

        batch_size = x_a.size(0)
        d = x_a.size(1)

        x = torch.zeros(batch_size * 3, d, device=DEVICE)
        x[0::3] = x_a
        x[1::3] = x_v
        x[2::3] = x_c

        base_edges = data.edge_index[:, :9]
        edge_index_list = []
        for i in range(batch_size):
            edge_index_list.append(base_edges + i * 3)
        edge_index = torch.cat(edge_index_list, dim=1).to(DEVICE)

        out = self.gat1(x, edge_index)
        out = F.gelu(out)
        out = self.gat2(out, edge_index)

        batch_vec = torch.arange(batch_size, device=DEVICE).view(-1, 1).repeat(1, 3).view(-1)
        out = global_mean_pool(out, batch_vec)

        logits = self.fc(out)
        return logits.squeeze(-1)

In [6]:
# ===============================
# 5. Setup Optimizer, Scheduler & Loss
# ===============================
model = AudioVideoSarcasmGATv2(
    a_dim=a_features.shape[1],
    v_dim=v_features.shape[1],
    c_dim=c_features.shape[1],
    hidden_dim=256,
    gat_dropout=0.5,
    fc_dropout=0.5,
).to(DEVICE)

decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(nd in name for nd in ["bias", "LayerNorm"]):
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)

criterion = nn.BCEWithLogitsLoss()

In [7]:
# ===============================
# 6. Train & Eval Functions
# ===============================
def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for data in loader:
        data = data.to(DEVICE)
        optimizer.zero_grad()

        logits = model(data)
        loss = criterion(logits, data.y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.y.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_preds.extend(preds)
        all_labels.extend(labels)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for data in loader:
        data = data.to(DEVICE)
        logits = model(data)

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return acc, f1, prec, rec, auc

In [8]:
# ===============================
# 7. Training Loop
# ===============================
best_val_f1 = 0.0
best_state  = None
patience_counter = 0

train_losses, val_f1s = [], []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = train_one_epoch(train_loader)
    val_acc, val_f1, val_prec, val_rec, val_auc = evaluate(val_loader)

    scheduler.step(epoch)

    train_losses.append(train_loss)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} F1 {train_f1:.4f} | "
        f"Val Acc {val_acc:.4f} F1 {val_f1:.4f} Prec {val_prec:.4f} Rec {val_rec:.4f} AUC {val_auc:.4f} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print(f"Training complete. Best Val F1: {best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(BASE_PATH, "best_sarcasm_gatv2_audio_video.pth"))
    print("Best model saved.")

Epoch 001 | Train Loss 0.6950 Acc 0.4668 F1 0.4636 | Val Acc 0.5000 F1 0.3333 Prec 0.2500 Rec 0.5000 AUC 0.7234 | LR 0.000049
Epoch 002 | Train Loss 0.6960 Acc 0.4855 F1 0.4762 | Val Acc 0.6731 F1 0.6711 Prec 0.6773 Rec 0.6731 AUC 0.7644 | LR 0.000045
Epoch 003 | Train Loss 0.6874 Acc 0.5602 F1 0.5598 | Val Acc 0.7019 F1 0.6916 Prec 0.7330 Rec 0.7019 AUC 0.7703 | LR 0.000040
Epoch 004 | Train Loss 0.6712 Acc 0.6120 F1 0.6106 | Val Acc 0.6538 F1 0.6308 Prec 0.7051 Rec 0.6538 AUC 0.7947 | LR 0.000033
Epoch 005 | Train Loss 0.6371 Acc 0.6722 F1 0.6722 | Val Acc 0.6538 F1 0.6406 Prec 0.6806 Rec 0.6538 AUC 0.7922 | LR 0.000025
Epoch 006 | Train Loss 0.6138 Acc 0.6950 F1 0.6947 | Val Acc 0.6538 F1 0.6455 Prec 0.6699 Rec 0.6538 AUC 0.7711 | LR 0.000017
Epoch 007 | Train Loss 0.6035 Acc 0.6909 F1 0.6909 | Val Acc 0.7500 F1 0.7496 Prec 0.7515 Rec 0.7500 AUC 0.7844 | LR 0.000010
Epoch 008 | Train Loss 0.5539 Acc 0.7593 F1 0.7593 | Val Acc 0.6635 F1 0.6609 Prec 0.6685 Rec 0.6635 AUC 0.7751 | LR 0

In [9]:
# ===============================
# 8. Final Test Evaluation
# ===============================
test_acc, test_f1, test_prec, test_rec, test_auc = evaluate(test_loader)
print(f"TEST PERFORMANCE | Acc {test_acc:.4f} F1 {test_f1:.4f} Prec {test_prec:.4f} Rec {test_rec:.4f} AUC {test_auc:.4f}")

TEST PERFORMANCE | Acc 0.7500 F1 0.7492 Prec 0.7534 Rec 0.7500 AUC 0.8095
